# 04 — Hyperparameter Tuning
**CMPE 188 | Flight Delay Prediction**

Goals:
- Load the enriched dataset (from notebook 02)
- Run GridSearchCV on XGBoost — completes the TODO from `scripts/xgboost_pipeline.py`
- Run RandomizedSearchCV on Random Forest
- 5-fold stratified cross-validation throughout
- Compare tuned models against notebook 03 baselines

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

DATA_PATH = "../data/processed/Airlines_enriched.csv"
df = pd.read_csv(DATA_PATH)

# Drop non-predictive columns
df = df.drop(columns=["id", "Flight"])

# --- Derived features (same as notebook 03) ---
hours = df["Time"] / 60
df["time_bucket"] = pd.cut(
    hours, bins=[0, 6, 12, 18, 24],
    labels=["night", "morning", "afternoon", "evening"], right=False,
).astype(str)
df["is_peak_hour"] = ((hours >= 7) & (hours < 9) | (hours >= 17) & (hours < 20)).astype(int)
df["route_volume"] = df.groupby(["AirportFrom", "AirportTo"])["Time"].transform("count")

print(f"Loaded {len(df):,} rows, {df.shape[1]} columns")
df.head()

## 1. Preprocessing + Train/Test Split

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

target = "Delay"
X = df.drop(columns=target)
y = df[target]

# --- Train/Test split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Airline delay rate (train-only to prevent leakage)
airline_delay_rate = X_train.join(y_train).groupby("Airline")[target].mean()
global_rate = y_train.mean()
X_train = X_train.copy()
X_test = X_test.copy()
X_train["airline_delay_rate"] = X_train["Airline"].map(airline_delay_rate).fillna(global_rate)
X_test["airline_delay_rate"] = X_test["Airline"].map(airline_delay_rate).fillna(global_rate)

# --- Preprocessing (includes weather + geo features from enriched dataset) ---
categorical_cols = ["Airline", "AirportFrom", "AirportTo", "time_bucket"]
numeric_cols = [
    "DayOfWeek", "Time", "Length", "is_peak_hour", "route_volume", "airline_delay_rate",
    # geo features
    "from_lat", "from_lon", "from_elevation_ft",
    "to_lat", "to_lon", "to_elevation_ft",
    # weather features
    "from_avg_temperature", "from_avg_precipitation", "from_avg_wind_speed",
    "to_avg_temperature", "to_avg_precipitation", "to_avg_wind_speed",
]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols),
        ("num", MinMaxScaler(), numeric_cols),
    ]
)

selector = SelectKBest(score_func=chi2, k=50)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Categorical: {categorical_cols}")
print(f"Numeric ({len(numeric_cols)}): {numeric_cols}")

## 2. GridSearchCV — XGBoost

In [ ]:
# TODO: define param_grid for n_estimators, max_depth, learning_rate, subsample
# TODO: GridSearchCV with StratifiedKFold(n_splits=5), scoring='roc_auc'
# TODO: print best_params_, best_score_

## 3. RandomizedSearchCV — Random Forest

In [ ]:
# TODO: define param_distributions for n_estimators, max_depth, min_samples_split, max_features
# TODO: RandomizedSearchCV with n_iter=20, StratifiedKFold(n_splits=5), scoring='roc_auc'
# TODO: print best_params_, best_score_

## 4. Baseline vs Tuned Comparison Table

In [ ]:
# Evaluate tuned models on test set
xgb_pred = xgb_grid.predict(X_test)
xgb_proba = xgb_grid.predict_proba(X_test)[:, 1]
xgb_tuned_acc = accuracy_score(y_test, xgb_pred)
xgb_tuned_auc = roc_auc_score(y_test, xgb_proba)

rf_pred = rf_random.predict(X_test)
rf_proba = rf_random.predict_proba(X_test)[:, 1]
rf_tuned_acc = accuracy_score(y_test, rf_pred)
rf_tuned_auc = roc_auc_score(y_test, rf_proba)

print("XGBoost Tuned Classification Report:")
print(classification_report(y_test, xgb_pred, target_names=["No Delay", "Delayed"]))
print("\nRandom Forest Tuned Classification Report:")
print(classification_report(y_test, rf_pred, target_names=["No Delay", "Delayed"]))

results = pd.DataFrame({
    "Model": ["XGBoost baseline", "XGBoost tuned", "RF baseline", "RF tuned"],
    "Features": ["raw + derived", "raw + derived + weather", "raw + derived", "raw + derived + weather"],
    "ROC-AUC": [0.6895, xgb_tuned_auc, 0.6848, rf_tuned_auc],
    "Accuracy": [0.6438, xgb_tuned_acc, 0.6384, rf_tuned_acc],
})
print("\n", results.to_string(index=False))